# Day 16 · 数据摄取

> **前置**:Day11-15 已掌握 NumPy / Pandas / 可视化
> **关键问题**:数据从哪里来?CSV、JSON、数据库是最常见的三种源头,本节学会用 Pandas 把它们全部读进 DataFrame,再把处理结果导回去。

**引入(5 分钟)**

上一节学会了"看"数据(`df.head()` / `df.describe()`),这一节学"取"数据。真实项目中,数据可能是一个 CSV 导出文件、一个接口返回的 JSON、一张数据库表。今天的任务就是掌握 **读入** 与 **导出** 两套工具链。

---

**第 1 讲:read_csv 参数详解**

**read_csv** 是最常用的读入函数,几乎每个项目都会遇到。本节重点掌握 **encoding**(文件编码,中文 CSV 常用 utf-8 或 gbk)、**sep**(分隔符,默认逗号,制表符用 `\t`)、**header**(第几行做列名)、**index_col**(指定某列做行索引)、**usecols**(只读部分列,节省内存)、**nrows**(只读前 N 行)、**skiprows**(跳过若干行)、**na_values**(哪些值视为缺失值)。


In [ ]:
import pandas as pd

# 用硬编码 DataFrame 模拟一份学生数据
df_demo = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, 21, 19, 22],
    "城市": ["北京", "上海", "广州", "深圳"],
})

# 先把它写出到临时 CSV,再用 read_csv 读回
# 真实项目中这里的 "students.csv" 就是真实文件路径
df_demo.to_csv("students.csv", index=False,
               encoding="utf-8")
df = pd.read_csv("students.csv", encoding="utf-8")
print("--- read_csv 读回 ---")
print(df)

# sep 参数:以分号做分隔符
df_demo.to_csv("students_sep.csv", index=False,
              sep=";", encoding="utf-8")
# 读取时必须指定 sep=";",否则整行都会变成一列
df_sep = pd.read_csv("students_sep.csv", sep=";",
                     encoding="utf-8")
print("\n--- sep=';' ---")
print(df_sep)

# index_col 参数:把第 0 列(姓名)设为行索引
df_idx = pd.read_csv("students.csv",
                     index_col=0, encoding="utf-8")
print("\n--- index_col=0 ---")
print(df_idx)

# 构造含缺失值的 CSV
df_na = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, None, 19, 22],
    "成绩": [88.5, 92.0, None, 76.5],
})
df_na.to_csv("students_na.csv", index=False,
             encoding="utf-8")

# usecols:只取姓名和成绩两列,省内存
df_cols = pd.read_csv("students_na.csv",
                      usecols=["姓名", "成绩"],
                      encoding="utf-8")
print("\n--- usecols ---")
print(df_cols)

# nrows:只读前 2 行(常用于预览大文件)
df_n = pd.read_csv("students_na.csv",
                   nrows=2, encoding="utf-8")
print("\n--- nrows=2 ---")
print(df_n)

# skiprows:跳过第 2 行(索引为 1 的行)
df_skip = pd.read_csv("students_na.csv",
                      skiprows=[1], encoding="utf-8")
print("\n--- skiprows=[1] ---")
print(df_skip)

# na_values:把 "NA" 和 "-" 也视为 NaN
df_na2 = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, "NA", "-"],
})
df_na2.to_csv("students_na2.csv", index=False,
              encoding="utf-8")
df_na_vals = pd.read_csv("students_na2.csv",
                         na_values=["NA", "-"],
                         encoding="utf-8")
print('\n--- na_values=["NA","-"] ---')
print(df_na_vals)


---

**第 2 讲:read_json + read_sql (SQLite)**

**read_json** 支持 JSON 字符串或 JSON 文件路径。最常用的是 **orient="records"**(列表套字典)。中文场景导出时务必加 **force_ascii=False**,否则中文会被转义成 `\uXXXX`。

**read_sql** 配合 SQLite 数据库使用。标准流程:**sqlite3.connect()** 建立连接 → 创建表、插入数据 → **pd.read_sql(query, conn)** 把查询结果读成 DataFrame → 最后用 **conn.close()** 关闭连接,或者用 **with** 上下文管理器自动关闭。


In [ ]:
import pandas as pd
import sqlite3

# ===== read_json =====
# 利用 DataFrame 构造 JSON 字符串(orient="records")
df_j = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19],
})
json_str = df_j.to_json(orient="records",
                        force_ascii=False)
print("--- 导出的 JSON 字符串 ---")
print(json_str)

# 再把 JSON 字符串读回 DataFrame
df_back = pd.read_json(json_str)
print("\n--- read_json 读回 ---")
print(df_back)

# orient 参数的三种取值
df_o = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21],
}, index=["a", "b"])
# orient="records":最通用,列表套字典
print('\norient="records"')
print(df_o.to_json(orient="records",
                   force_ascii=False))
# orient="index":外层键是行索引
print('\norient="index"')
print(df_o.to_json(orient="index",
                   force_ascii=False))

# ===== read_sql (SQLite) =====
# 创建内存数据库(程序结束自动销毁)
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
# 创建 students 表
cur.execute(
    "CREATE TABLE students "
    "(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)"
)
# 批量插入数据
rows = [(1, "张三", 20), (2, "李四", 21),
        (3, "王五", 19), (4, "赵六", 22)]
cur.executemany(
    "INSERT INTO students VALUES (?, ?, ?)", rows
)
conn.commit()
# 条件查询 age>20,读成 DataFrame
df_sql = pd.read_sql(
    "SELECT * FROM students WHERE age > 20", conn
)
print("\n--- 条件查询 age>20 ---")
print(df_sql)
# 关闭连接(必须调用,否则可能锁表)
conn.close()

# 用 with 上下文管理器(更安全,自动关闭)
with sqlite3.connect(":memory:") as conn2:
    cur2 = conn2.cursor()
    cur2.execute(
        "CREATE TABLE books (title TEXT, price REAL)"
    )
    cur2.executemany(
        "INSERT INTO books VALUES (?, ?)",
        [("Python基础", 59.8), ("数据分析", 88.0)],
    )
    conn2.commit()
    df_books = pd.read_sql(
        "SELECT * FROM books", conn2
    )
    print("\n--- with 上下文读取 books ---")
    print(df_books)
# with 块结束,conn2 自动关闭


---

**第 3 讲:数据导出 to_csv + to_json**

分析完成后常需把结果持久化。**to_csv** 默认会把行索引也写成单独一列,通常不需要,所以必须加 **index=False**,并指定 **encoding** 防止中文乱码。

**to_json** 最常用的是 **orient="records"**(列表套字典,其他语言读起来最方便),配合 **force_ascii=False** 保留中文,加 **indent=2** 美化格式。


In [ ]:
import pandas as pd

# ===== to_csv =====
# 构造一份数据
df_out = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19],
    "成绩": [88.5, 92.0, 76.5],
})
# 默认导出:会多出一列行索引(0,1,2)
df_out.to_csv("with_index.csv", encoding="utf-8")
print("--- 默认导出(含行索引) ---")
print(pd.read_csv("with_index.csv"))

# index=False:去掉行索引,更干净
df_out.to_csv("no_index.csv", index=False,
              encoding="utf-8")
print("\n--- index=False 导出 ---")
print(pd.read_csv("no_index.csv"))

# ===== to_json =====
df_j = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21],
})
# orient="records" + force_ascii=False + indent=2
df_j.to_json("pretty.json", orient="records",
             force_ascii=False, indent=2)
print("\n--- 已写出 pretty.json ---")
# 读回验证
df_back = pd.read_json("pretty.json")
print(df_back)

# 对比:force_ascii=True(默认)会把中文转义
print("\n--- force_ascii=True(默认) ---")
print(df_j.to_json(orient="records"))
print("\n--- force_ascii=False ---")
print(df_j.to_json(orient="records",
                   force_ascii=False))


---

**今日小结**

**pd.read_csv** 读 CSV,常用参数 encoding / sep / header / index_col / usecols / nrows / skiprows / na_values;**pd.read_json** 读 JSON,orient="records" 最常用;**pd.read_sql** 读 SQL,需要 sql 语句 + conn 连接对象;**df.to_csv** 写 CSV,加 index=False;**df.to_json** 写 JSON,orient="records" + force_ascii=False。

**更多练习**

- 当堂练:course/lesson16/in_class/practice01-06.py
- 课后作业:course/lesson16/homework/task01-03.py
